In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from ast import literal_eval
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.optimize import minimize
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import os

In [30]:
# Define helper functions
def load_csv(filename):
    return pd.read_csv(filename, sep=',')

def calculate_group_average(df):
    return df.groupby(['Label', 'Conc']).mean()

def transform_label(label):
    # Check for condition 1: Replace specific strings with 'Reference'
    if label in (["['AF']", "['AgNR@SiO2']", "['DMEM']"]):
        return "['Reference']"
    
    # Check for condition 2: Replace 'COVNL63' or 'CoVNL63' with 'CoVNL63'
    if 'COVNL63' in label or 'CoVNL63' in label:
        label = label.replace('COVNL63', 'CoVNL63')
        label = label.replace('CoVNL63', 'CoVNL63')
        label = label.replace('CovNL63', 'CoVNL63')
    
    return label

def inspect_dataframe(df):
    """
    Inspects a DataFrame to understand its structure, particularly the Label and Conc columns.
    """
    print("DataFrame shape:", df.shape)
    print("\nColumn names:", df.columns.tolist())
    
    # Inspect index structure
    print("\nIndex type:", type(df.index))
    print("First 5 index values:", df.index[:5])
    
    if 'Label' in df.columns or ('Label' in df.index.names):
        # Get unique labels
        if 'Label' in df.columns:
            labels = df['Label'].unique()
        else:
            # For MultiIndex
            labels = df.index.get_level_values('Label').unique()
        
        print("\nUnique Label values (first 10):")
        for i, label in enumerate(labels[:10]):
            print(f"{i+1}. {repr(label)} (Type: {type(label)})")
    
    if 'Conc' in df.columns or ('Conc' in df.index.names):
        # Get unique concentrations
        if 'Conc' in df.columns:
            concs = df['Conc'].unique()
        else:
            # For MultiIndex
            concs = df.index.get_level_values('Conc').unique()
        
        print("\nUnique Conc values (first 10):")
        for i, conc in enumerate(concs[:10]):
            print(f"{i+1}. {repr(conc)} (Type: {type(conc)})")
    
    # Check for problematic mixed virus entries
    if isinstance(df.index, pd.MultiIndex) and 'Label' in df.index.names:
        print("\nChecking for mixed virus entries:")
        for idx in df.index:
            label = idx[df.index.names.index('Label')]
            if "__" in str(label):
                print(f"Found mixed virus: {repr(label)}")
                # Print the full row for this index
                print(f"  Row data: {idx}")
                break

def parse_mixed_virus_data(fixed_df):
    """
    Parse virus names and concentrations from mixed virus data.
    Uses ast.literal_eval to safely parse the Label strings as Python literals,
    ensuring accurate virus type detection.
    """
    import ast  # For safely evaluating the label as a Python literal
    
    # Create dictionary to store parsed results
    parsed_data = {}
    
    # Reset index to access the Label and Conc columns
    fixed_df_reset = fixed_df.reset_index()
    
    # Process each row
    for idx, row in fixed_df_reset.iterrows():
        label = row['Label']
        conc = row['Conc']
        
        # Parse the virus names from the label using literal_eval
        virus_names = []
        if isinstance(label, str):
            try:
                # Parse the label as a Python literal (usually a list of strings)
                parsed_label = ast.literal_eval(label)
                
                if isinstance(parsed_label, list):
                    # Successfully parsed as a list
                    virus_names = parsed_label
                else:
                    # Not a list, just use the single item
                    virus_names = [str(parsed_label)]
            except (SyntaxError, ValueError):
                # If parsing fails, fall back to simple string handling
                clean_label = label.strip("[]' ")
                virus_names = [clean_label]
        
        # Parse the concentrations from the conc - for clean CSV format with brackets
        concentrations = []
        if isinstance(conc, str):
            try:
                # Try to parse the concentration as a Python literal first
                parsed_conc = ast.literal_eval(conc)
                
                if isinstance(parsed_conc, list):
                    # It's a list of concentrations
                    concentrations = [float(c) for c in parsed_conc]
                else:
                    # Single concentration
                    concentrations = [float(parsed_conc)]
            except (SyntaxError, ValueError):
                # If parsing fails, fall back to string splitting
                # First, remove any surrounding brackets if present
                clean_conc = conc.strip()
                if clean_conc.startswith('[') and clean_conc.endswith(']'):
                    clean_conc = clean_conc[1:-1]  # Remove the brackets
                
                # Handle the format with comma separators
                if "," in clean_conc:
                    # Split by comma and convert each part to float
                    parts = clean_conc.split(",")
                    for part in parts:
                        part = part.strip()
                        if part:  # Skip empty strings
                            try:
                                concentrations.append(float(part))
                            except ValueError:
                                print(f"Warning: Could not convert '{part}' to float")
                else:
                    # No separators, just a single value
                    if clean_conc:
                        try:
                            concentrations.append(float(clean_conc))
                        except ValueError:
                            print(f"Warning: Could not convert '{clean_conc}' to float")
        elif isinstance(conc, (int, float)):
            concentrations.append(conc)
        
        # Determine the type of mixture
        if len(virus_names) == 1:
            parsed_type = "single"
        elif len(virus_names) == 2:
            parsed_type = "double"
        elif len(virus_names) == 3:
            parsed_type = "triple"
        else:
            parsed_type = f"multiple({len(virus_names)})"
        
        # Store the parsed data using the original index as key
        original_idx = (label, conc)
        parsed_data[original_idx] = {
            "type": parsed_type,
            "virus_names": virus_names,
            "concentrations": concentrations
        }
    
    return parsed_data

def get_single_virus_highest_concentration_rows(avg_df):
    """
    Get the highest concentration row for each single virus (excluding mixed virus samples).
    Updated to handle the clean CSV format with bracketed values.
    
    Args:
        avg_df: DataFrame with virus data, may include mixed virus samples
        
    Returns:
        DataFrame with highest concentration row for each single virus
    """
    avg_df_reset = avg_df.reset_index()
    
    print("Extracting single virus data...")
    
    # Filter for single virus labels only
    single_virus_mask = avg_df_reset['Label'].apply(
        lambda x: isinstance(x, str) and 
                 not '__' in x and  # Skip mixed virus labels with '__'
                 len(x.strip("[]").replace("'", "").split(", ")) == 1
    )
    
    single_virus_df = avg_df_reset[single_virus_mask].copy()
    print(f"Found {len(single_virus_df)} single virus samples")
    
    if single_virus_df.empty:
        print("No single virus samples found!")
        return pd.DataFrame()
    
    # Print some example concentration values for debugging
    print("Example concentration values:")
    print(single_virus_df['Conc'].head(5).to_list())
    
    # Parse concentration values for single virus samples - Updated for clean CSV format with brackets
    def parse_conc(x):
        try:
            if isinstance(x, str):
                # First, remove any surrounding brackets if present
                clean_str = x.strip()
                if clean_str.startswith('[') and clean_str.endswith(']'):
                    clean_str = clean_str[1:-1]  # Remove the first and last characters (brackets)
                
                # Handle new format with comma separators
                if "," in clean_str:
                    # For single virus, just take the first value before comma
                    first_value = clean_str.split(",")[0].strip()
                    return float(first_value)
                else:
                    # Just try to convert to float directly
                    return float(clean_str)
            elif isinstance(x, list):
                # If it's already a list, return the first element
                return float(x[0])
            else:
                # If it's a scalar value, return it as is
                return float(x)
        except Exception as e:
            print(f"Error parsing concentration: {x}, Error: {e}")
            # Return the original value if parsing fails
            return x
    
    # Apply the parsing function
    single_virus_df['Conc_parsed'] = single_virus_df['Conc'].apply(parse_conc)
    
    # For debugging
    print("After parsing concentrations:")
    print(single_virus_df[['Label', 'Conc', 'Conc_parsed']].head())
    
    # Get unique single virus labels
    unique_virus_labels = single_virus_df['Label'].unique()
    print(f"Found {len(unique_virus_labels)} unique single virus types")
    
    # Get highest concentration for each virus
    highest_conc_rows = []
    for label in unique_virus_labels:
        label_rows = single_virus_df[single_virus_df['Label'] == label]
        if not label_rows.empty:
            try:
                # Sort by parsed concentration and get highest
                highest_conc_row = label_rows.sort_values(by='Conc_parsed', ascending=False).iloc[0]
                # Remove the temporary Conc_parsed column before adding to results
                highest_conc_row = highest_conc_row.drop('Conc_parsed')
                highest_conc_rows.append(highest_conc_row)
                print(f"Highest concentration for {label}: {highest_conc_row['Conc']}")
            except Exception as e:
                print(f"Error processing {label}: {str(e)}")
    
    result_df = pd.DataFrame(highest_conc_rows)
    print(f"Final result has {len(result_df)} rows")
    return result_df
    
def fit_least_squares(components, target, lam=1.0, regularization="None"):
    X = np.column_stack(components)
    y = target.values.reshape(-1, 1).ravel()
    num_vars = X.shape[1]
    
    # Objective function: Sum of squared residuals with regularization
    def objective(coefs):
        predictions = X @ coefs
        residuals = y - predictions
        reg_term = 0
        if regularization == "ridge":
            reg_term = lam * np.sum(coefs**2)
        elif regularization == "lasso":
            reg_term = lam * np.sum(np.abs(coefs))
        elif regularization == "None":
            reg_term = 0
        return np.sum(residuals**2) + reg_term
    
    # Constraints: Coefficients sum to 1
    cons = {'type': 'eq', 'fun': lambda coefs: np.sum(coefs) - 1}
    
    # Bounds: Coefficients are non-negative
    bounds = [(1e-10, 1) for _ in range(num_vars)]
    
    # Initial guess: all ones
    initial_coefs = np.ones(num_vars)
    
    # Perform optimization
    result = minimize(objective, initial_coefs, bounds=bounds, constraints=cons)
    
    if not result.success:
        raise ValueError("Constrained optimization failed!")
    
    coefficients = result.x
    
    return coefficients, 0

def process_dataframe(fixed_df, parsed_data, highest_single_virus_conc_df, reference_series):
    """
    Process the dataframe and return predicted spectra and fit results.
    """
    # Initialize DataFrame to store results
    predicted_df = pd.DataFrame()
    result_df = pd.DataFrame(columns=['Label', 'Conc', 'Coefficients', 'Intercept', 'MAE', 'RMSE', 'R2'])
    
    # Process each sample
    for (label, conc), target_avg in fixed_df.iterrows():
        if 'Reference' not in str(label):  # Skip reference samples
            print(f"\nProcessing sample: Label={label}, Conc={conc}")
            
            try:
                # Get the parsed virus names and concentrations
                parsed_idx = (label, conc)
                if parsed_idx in parsed_data:
                    virus_data = parsed_data[parsed_idx]
                    virus_names = virus_data["virus_names"]
                    concentrations = virus_data["concentrations"]
                else:
                    print(f"Warning: No parsed data found for {parsed_idx}")
                    continue
                    
                print(f"Parsed viruses: {virus_names}")
                print(f"Parsed concentrations: {concentrations}")
                
                # Get the average values for each of the single-virus components
                components = []
                for virus_name in virus_names:
                    # Format the virus name for lookup
                    formatted_virus = f"['{virus_name}']"
                    matching_rows = highest_single_virus_conc_df[highest_single_virus_conc_df['Label'] == formatted_virus]
                    
                    if matching_rows.empty:
                        print(f"Warning: No data found for virus '{virus_name}'")
                        continue
                    
                    # Get the highest concentration data for this virus
                    virus_component = matching_rows.iloc[0, 2:].astype(float)
                    components.append(virus_component)
                
                # Add reference component
                components.append(reference_series)
                
                if len(components) < 2:
                    print(f"Warning: Not enough components found for label {label}")
                    continue
                
                # Perform least squares fitting
                coefficients, intercept = fit_least_squares(components, target_avg)
                print(f"Fitted coefficients: {coefficients}")
                
                # Compute predicted values
                predicted_avg = sum(comp * coef for comp, coef in zip(components, coefficients)) + intercept
                
                # Compute MAE, RMSE, and R2
                mae = mean_absolute_error(target_avg, predicted_avg)
                rmse = np.sqrt(mean_squared_error(target_avg, predicted_avg))
                r2 = r2_score(target_avg, predicted_avg)
                
                # Compute VIFs for predictors
                X_with_const = add_constant(np.column_stack(components))
                
                # Handle NaN or infinite values
                for i, comp in enumerate(components):
                    if not np.isfinite(comp).all():
                        print(f"Warning: Component {i} contains NaNs or infinities")
                        components[i] = comp.fillna(0).replace([np.inf, -np.inf], 0)
                
                # Calculate VIF for each predictor
                vifs = [variance_inflation_factor(X_with_const, i) for i in range(X_with_const.shape[1])]
                vifs = vifs[1:]  # Exclude constant term
                
                # Append results to DataFrame
                data_dict = predicted_avg.to_dict()
                data_dict['Label'] = label
                data_dict['Conc'] = conc
                
                # Add to predicted_df
                predicted_df = pd.concat([predicted_df, pd.DataFrame([data_dict])], ignore_index=True)
                
                # Add to result_df
                result_row = {
                    'Label': label, 
                    'Conc': conc, 
                    'Coefficients': coefficients, 
                    'Intercept': intercept, 
                    'MAE': mae, 
                    'RMSE': rmse, 
                    'R2': r2, 
                    'VIFs': vifs
                }
                result_df = pd.concat([result_df, pd.DataFrame([result_row])], ignore_index=True)
                
            except Exception as e:
                print(f"Error processing label={label}, conc={conc}: {str(e)}")
                import traceback
                traceback.print_exc()
                # Continue to the next sample
                continue
    
    return predicted_df, result_df

In [22]:
# Set up the paths and folders
WORKING_PATH = "/home/zhao/Jiaheng Cui/Coefficient_fitting"
DATA_FOLDER = WORKING_PATH + "/data/linear decomposition"
DATE = "04042025"
RESULT_FOLDER = WORKING_PATH + f"/{DATE} - reconstructed vs real only using GLF"
os.makedirs(RESULT_FOLDER, exist_ok=True)

In [ ]:
# Step 1: Load CSV data
filename = DATA_FOLDER + "/test (virus in water).csv"
df = pd.read_csv(filename, sep=',')

# Apply label transformation
df['Label'] = df['Label'].apply(transform_label)
df

,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,458.0,459.0,...,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0,Label,Conc
0,0.170,0.171,0.175,0.181,0.192,0.203,0.214,0.223,0.232,0.240,...,0.682,0.664,0.649,0.636,0.624,0.614,0.602,0.589,['Ad5'],[100.0]
1,0.143,0.147,0.152,0.157,0.163,0.170,0.178,0.186,0.193,0.203,...,0.679,0.664,0.653,0.642,0.631,0.620,0.607,0.593,['Ad5'],[100.0]
2,0.168,0.168,0.172,0.179,0.194,0.205,0.216,0.224,0.228,0.230,...,0.663,0.646,0.630,0.615,0.601,0.589,0.576,0.563,['Ad5'],[100.0]
3,0.191,0.191,0.193,0.197,0.205,0.214,0.225,0.236,0.248,0.263,...,0.699,0.682,0.661,0.644,0.630,0.623,0.611,0.596,['Ad5'],[100.0]
4,0.192,0.199,0.205,0.209,0.209,0.211,0.213,0.216,0.221,0.229,...,0.674,0.657,0.642,0.626,0.610,0.594,0.579,0.565,['Ad5'],[100.0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88699,0.124,0.148,0.160,0.169,0.174,0.175,0.165,0.167,0.173,0.181,...,0.638,0.623,0.621,0.611,0.596,0.569,0.547,0.531,['RSVB1'],[781.0]
88700,0.052,0.066,0.072,0.075,0.075,0.071,0.061,0.061,0.064,0.072,...,0.688,0.678,0.676,0.667,0.655,0.631,0.614,0.604,['RSVB1'],[781.0]
88701,0.140,0.158,0.170,0.179,0.185,0.188,0.185,0.187,0.191,0.196,...,0.699,0.688,0.682,0.674,0.666,0.656,0.647,0.637,['RSVB1'],[781.0]
88702,0.153,0.161,0.163,0.160,0.152,0.141,0.121,0.118,0.123,0.133,...,0.631,0.623,0.621,0.616,0.608,0.595,0.582,0.570,['RSVB1'],[781.0]


In [24]:
# Group and save the grouped data
grouped_df = df.groupby(['Label', 'Conc'], as_index=False).mean()
grouped_df

,Label,Conc,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,...,1691.0,1692.0,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0
0,['Ad5'],[100.0],0.188456,0.193579,0.199298,0.205842,0.214035,0.222088,0.230509,0.238614,...,0.724246,0.706123,0.688772,0.672316,0.656649,0.642035,0.628474,0.616544,0.603561,0.589719
1,['Ad5'],[100000.0],0.108017,0.115627,0.123576,0.132051,0.140915,0.149695,0.160068,0.166644,...,1.058237,1.045763,1.033407,1.020407,1.007390,0.992627,0.976119,0.957441,0.937000,0.914610
2,['Ad5'],[12500.0],0.125390,0.136017,0.145695,0.155153,0.164305,0.172593,0.181102,0.186508,...,1.031186,1.019051,1.007000,0.994695,0.982814,0.969153,0.953915,0.936441,0.918119,0.898695
3,['Ad5'],[1562.0],0.144458,0.155678,0.165746,0.175237,0.184034,0.191864,0.199475,0.204407,...,1.011746,1.001780,0.990966,0.979593,0.967610,0.954356,0.939966,0.923898,0.907797,0.891576
4,['Ad5'],[195.0],0.142603,0.152603,0.161483,0.169983,0.177655,0.184931,0.192069,0.197793,...,0.880224,0.866776,0.852776,0.838741,0.824276,0.809707,0.795000,0.779914,0.764931,0.750431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2575,['RSVB1'],[50.0],0.136810,0.138224,0.141741,0.146172,0.151672,0.158603,0.166310,0.174000,...,0.744603,0.726397,0.707000,0.688672,0.671207,0.653741,0.637914,0.623000,0.611241,0.598483
2576,['RSVB1'],[50000.0],0.154390,0.162458,0.167458,0.171000,0.172881,0.173220,0.170322,0.170288,...,0.860475,0.851661,0.843661,0.836492,0.831085,0.823475,0.813983,0.801017,0.789203,0.778390
2577,['RSVB1'],[6250.0],0.177224,0.182293,0.185483,0.187879,0.189190,0.189483,0.187948,0.187966,...,0.815948,0.802086,0.791810,0.783845,0.781293,0.775534,0.767638,0.754776,0.744207,0.735483
2578,['RSVB1'],[781.0],0.126759,0.135397,0.141224,0.145879,0.149379,0.151500,0.151138,0.153155,...,0.735293,0.720603,0.709017,0.699397,0.694690,0.686190,0.675052,0.658069,0.643534,0.630948


In [25]:
pd.DataFrame(grouped_df).to_csv(DATA_FOLDER + f"/New folder/{DATE} - average_spectra_grouped_test_only_GLF_with_SiO2.csv", index=False)

In [26]:
# Calculate group average
avg_df = calculate_group_average(df)
print("Average DataFrame shape:", avg_df.shape)

# Inspect the DataFrame to understand its structure
inspect_dataframe(avg_df)
avg_df

Average DataFrame shape: (2580, 1251)
DataFrame shape: (2580, 1251)

Column names: ['450.0', '451.0', '452.0', '453.0', '454.0', '455.0', '456.0', '457.0', '458.0', '459.0', '460.0', '461.0', '462.0', '463.0', '464.0', '465.0', '466.0', '467.0', '468.0', '469.0', '470.0', '471.0', '472.0', '473.0', '474.0', '475.0', '476.0', '477.0', '478.0', '479.0', '480.0', '481.0', '482.0', '483.0', '484.0', '485.0', '486.0', '487.0', '488.0', '489.0', '490.0', '491.0', '492.0', '493.0', '494.0', '495.0', '496.0', '497.0', '498.0', '499.0', '500.0', '501.0', '502.0', '503.0', '504.0', '505.0', '506.0', '507.0', '508.0', '509.0', '510.0', '511.0', '512.0', '513.0', '514.0', '515.0', '516.0', '517.0', '518.0', '519.0', '520.0', '521.0', '522.0', '523.0', '524.0', '525.0', '526.0', '527.0', '528.0', '529.0', '530.0', '531.0', '532.0', '533.0', '534.0', '535.0', '536.0', '537.0', '538.0', '539.0', '540.0', '541.0', '542.0', '543.0', '544.0', '545.0', '546.0', '547.0', '548.0', '549.0', '550.0', '551.0'

450.0     451.0     452.0     453.0     454.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.188456  0.193579  0.199298  0.205842  0.214035   
              [100000.0]  0.108017  0.115627  0.123576  0.132051  0.140915   
              [12500.0]   0.125390  0.136017  0.145695  0.155153  0.164305   
              [1562.0]    0.144458  0.155678  0.165746  0.175237  0.184034   
              [195.0]     0.142603  0.152603  0.161483  0.169983  0.177655   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.136810  0.138224  0.141741  0.146172  0.151672   
              [50000.0]   0.154390  0.162458  0.167458  0.171000  0.172881   
              [6250.0]    0.177224  0.182293  0.185483  0.187879  0.189190   
              [781.0]     0.126759  0.135397  0.141224  0.145879  0.149379   
['Reference'] [0.0]       0.159350  0.167391  0.175635  0.184376  0.193345   

                             455.0     456.0     457.0     458.0     459.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.222088  0.230509  0.238614  0.246246  0.255404   
              [100000.0]  0.149695  0.160068  0.166644  0.171949  0.176271   
              [12500.0]   0.172593  0.181102  0.186508  0.190441  0.193508   
              [1562.0]    0.191864  0.199475  0.204407  0.207966  0.210695   
              [195.0]     0.184931  0.192069  0.197793  0.202759  0.208310   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.158603  0.166310  0.174000  0.182017  0.189966   
              [50000.0]   0.173220  0.170322  0.170288  0.170932  0.172356   
              [6250.0]    0.189483  0.187948  0.187966  0.188310  0.189276   
              [781.0]     0.151500  0.151138  0.153155  0.155931  0.159466   
['Reference'] [0.0]       0.202345  0.213487  0.219716  0.224497  0.228563   

                          ...    1691.0    1692.0    1693.0    1694.0  \
Label         Conc        ...                                           
['Ad5']       [100.0]     ...  0.724246  0.706123  0.688772  0.672316   
              [100000.0]  ...  1.058237  1.045763  1.033407  1.020407   
              [12500.0]   ...  1.031186  1.019051  1.007000  0.994695   
              [1562.0]    ...  1.011746  1.001780  0.990966  0.979593   
              [195.0]     ...  0.880224  0.866776  0.852776  0.838741   
...                       ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      ...  0.744603  0.726397  0.707000  0.688672   
              [50000.0]   ...  0.860475  0.851661  0.843661  0.836492   
              [6250.0]    ...  0.815948  0.802086  0.791810  0.783845   
              [781.0]     ...  0.735293  0.720603  0.709017  0.699397   
['Reference'] [0.0]       ...  0.884071  0.872670  0.861655  0.850949   

                            1695.0    1696.0    1697.0    1698.0    1699.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.656649  0.642035  0.628474  0.616544  0.603561   
              [100000.0]  1.007390  0.992627  0.976119  0.957441  0.937000   
              [12500.0]   0.982814  0.969153  0.953915  0.936441  0.918119   
              [1562.0]    0.967610  0.954356  0.939966  0.923898  0.907797   
              [195.0]     0.824276  0.809707  0.795000  0.779914  0.764931   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.671207  0.653741  0.637914  0.623000  0.611241   
              [50000.0]   0.831085  0.823475  0.813983  0.801017  0.789203   
              [6250.0]    0.781293  0.775534  0.767638  0.754776  0.744207   
              [781.0]     0.694690  0.686190  0.675052  0.658069  0.643534   
['Reference'] [0.0]       0.841188  0.829807  0.817056  0.801919  0.786701   

                            1700.0

In [27]:
# # Step 2: Fix mixed virus data format issues
# print("\nFixing mixed virus data...")
# fixed_df = fix_mixed_virus_data(avg_df)
# fixed_df

# FOR CLEAN CSV FORMAT, WE SKIP THE FIX STEP
# The data is already in the correct format, so we use avg_df directly
fixed_df = avg_df
fixed_df

450.0     451.0     452.0     453.0     454.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.188456  0.193579  0.199298  0.205842  0.214035   
              [100000.0]  0.108017  0.115627  0.123576  0.132051  0.140915   
              [12500.0]   0.125390  0.136017  0.145695  0.155153  0.164305   
              [1562.0]    0.144458  0.155678  0.165746  0.175237  0.184034   
              [195.0]     0.142603  0.152603  0.161483  0.169983  0.177655   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.136810  0.138224  0.141741  0.146172  0.151672   
              [50000.0]   0.154390  0.162458  0.167458  0.171000  0.172881   
              [6250.0]    0.177224  0.182293  0.185483  0.187879  0.189190   
              [781.0]     0.126759  0.135397  0.141224  0.145879  0.149379   
['Reference'] [0.0]       0.159350  0.167391  0.175635  0.184376  0.193345   

                             455.0     456.0     457.0     458.0     459.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.222088  0.230509  0.238614  0.246246  0.255404   
              [100000.0]  0.149695  0.160068  0.166644  0.171949  0.176271   
              [12500.0]   0.172593  0.181102  0.186508  0.190441  0.193508   
              [1562.0]    0.191864  0.199475  0.204407  0.207966  0.210695   
              [195.0]     0.184931  0.192069  0.197793  0.202759  0.208310   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.158603  0.166310  0.174000  0.182017  0.189966   
              [50000.0]   0.173220  0.170322  0.170288  0.170932  0.172356   
              [6250.0]    0.189483  0.187948  0.187966  0.188310  0.189276   
              [781.0]     0.151500  0.151138  0.153155  0.155931  0.159466   
['Reference'] [0.0]       0.202345  0.213487  0.219716  0.224497  0.228563   

                          ...    1691.0    1692.0    1693.0    1694.0  \
Label         Conc        ...                                           
['Ad5']       [100.0]     ...  0.724246  0.706123  0.688772  0.672316   
              [100000.0]  ...  1.058237  1.045763  1.033407  1.020407   
              [12500.0]   ...  1.031186  1.019051  1.007000  0.994695   
              [1562.0]    ...  1.011746  1.001780  0.990966  0.979593   
              [195.0]     ...  0.880224  0.866776  0.852776  0.838741   
...                       ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      ...  0.744603  0.726397  0.707000  0.688672   
              [50000.0]   ...  0.860475  0.851661  0.843661  0.836492   
              [6250.0]    ...  0.815948  0.802086  0.791810  0.783845   
              [781.0]     ...  0.735293  0.720603  0.709017  0.699397   
['Reference'] [0.0]       ...  0.884071  0.872670  0.861655  0.850949   

                            1695.0    1696.0    1697.0    1698.0    1699.0  \
Label         Conc                                                           
['Ad5']       [100.0]     0.656649  0.642035  0.628474  0.616544  0.603561   
              [100000.0]  1.007390  0.992627  0.976119  0.957441  0.937000   
              [12500.0]   0.982814  0.969153  0.953915  0.936441  0.918119   
              [1562.0]    0.967610  0.954356  0.939966  0.923898  0.907797   
              [195.0]     0.824276  0.809707  0.795000  0.779914  0.764931   
...                            ...       ...       ...       ...       ...   
['RSVB1']     [50.0]      0.671207  0.653741  0.637914  0.623000  0.611241   
              [50000.0]   0.831085  0.823475  0.813983  0.801017  0.789203   
              [6250.0]    0.781293  0.775534  0.767638  0.754776  0.744207   
              [781.0]     0.694690  0.686190  0.675052  0.658069  0.643534   
['Reference'] [0.0]       0.841188  0.829807  0.817056  0.801919  0.786701   

                            1700.0

In [31]:
# Step 3: Parse virus names and concentrations with updated function
print("\nParsing virus names and concentrations...")
parsed_data = parse_mixed_virus_data(fixed_df)
# Print summary of parsed data types
mixture_types = {}
for idx, data in parsed_data.items():
    mixture_type = data["type"]
    if mixture_type not in mixture_types:
        mixture_types[mixture_type] = 0
    mixture_types[mixture_type] += 1
print("\nSummary of parsed data:")
for mixture_type, count in mixture_types.items():
    print(f"  {mixture_type}: {count} samples")


Parsing virus names and concentrations...

Summary of parsed data:
  single: 150 samples
  double: 960 samples
  triple: 1470 samples


In [32]:
parsed_data

{("['Ad5']", '[100.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [100.0]},
 ("['Ad5']", '[100000.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [100000.0]},
 ("['Ad5']", '[12500.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [12500.0]},
 ("['Ad5']", '[1562.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [1562.0]},
 ("['Ad5']", '[195.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [195.0]},
 ("['Ad5']", '[25000.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [25000.0]},
 ("['Ad5']", '[3125.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [3125.0]},
 ("['Ad5']", '[391.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [391.0]},
 ("['Ad5']", '[50000.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'concentrations': [50000.0]},
 ("['Ad5']", '[6250.0]'): {'type': 'single',
  'virus_names': ['Ad5'],
  'conc

In [33]:
# Step 4: Get the highest concentration rows for single-virus labels
highest_single_virus_conc_df = get_single_virus_highest_concentration_rows(avg_df)
print("\nHighest concentration single virus DataFrame shape:", highest_single_virus_conc_df.shape)
highest_single_virus_conc_df

Extracting single virus data...
Found 150 single virus samples
Example concentration values:
['[100.0]', '[100000.0]', '[12500.0]', '[1562.0]', '[195.0]']
After parsing concentrations:
     Label        Conc  Conc_parsed
0  ['Ad5']     [100.0]        100.0
1  ['Ad5']  [100000.0]     100000.0
2  ['Ad5']   [12500.0]      12500.0
3  ['Ad5']    [1562.0]       1562.0
4  ['Ad5']     [195.0]        195.0
Found 14 unique single virus types
Highest concentration for ['Ad5']: [100000.0]
Highest concentration for ['CoV2']: [100000.0]
Highest concentration for ['CoV229E']: [100000.0]
Highest concentration for ['CoV2B1']: [100000.0]
Highest concentration for ['CoVNL63']: [100000.0]
Highest concentration for ['CoVOC43']: [100000.0]
Highest concentration for ['FluB']: [100000.0]
Highest concentration for ['H1N1']: [100000.0]
Highest concentration for ['H3N2']: [100000.0]
Highest concentration for ['HMPVA']: [100000.0]
Highest concentration for ['HMPVB']: [100000.0]
Highest concentration for ['RSVA2']

,Label,Conc,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,...,1691.0,1692.0,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0
1,['Ad5'],[100000.0],0.108017,0.115627,0.123576,0.132051,0.140915,0.149695,0.160068,0.166644,...,1.058237,1.045763,1.033407,1.020407,1.007390,0.992627,0.976119,0.957441,0.937000,0.914610
12,['CoV2'],[100000.0],0.143678,0.158237,0.170542,0.182068,0.192441,0.201322,0.209508,0.215186,...,0.970017,0.953678,0.940966,0.930339,0.924712,0.915695,0.904000,0.887271,0.871712,0.857068
23,['CoV229E'],[100000.0],0.026228,0.023702,0.023737,0.024930,0.027228,0.030421,0.036719,0.038281,...,1.206211,1.187544,1.167246,1.145368,1.122035,1.097088,1.070860,1.042912,1.014982,0.987281
35,['CoV2B1'],[100000.0],0.309544,0.322877,0.337579,0.353386,0.369930,0.386789,0.407053,0.419105,...,1.195842,1.181596,1.163491,1.141491,1.114737,1.087912,1.061368,1.034211,1.010965,0.990649
2036,['CoVNL63'],[100000.0],0.059492,0.065085,0.071356,0.077983,0.084492,0.090576,0.098661,0.100508,...,1.009051,1.005831,0.999797,0.991441,0.979644,0.967000,0.953153,0.938627,0.922085,0.903356
2048,['CoVOC43'],[100000.0],0.131847,0.137356,0.146068,0.156712,0.168932,0.182966,0.200000,0.214000,...,1.144441,1.152814,1.158136,1.161000,1.160203,1.157237,1.151763,1.143576,1.132695,1.118966
2060,['FluB'],[100000.0],0.109414,0.115172,0.122069,0.129431,0.137362,0.145586,0.154707,0.162293,...,1.108776,1.088431,1.070414,1.053431,1.040483,1.022914,1.001603,0.973517,0.946621,0.920534
2291,['H1N1'],[100000.0],0.055175,0.060404,0.066579,0.073368,0.080491,0.087807,0.096965,0.102035,...,1.770947,1.741158,1.709351,1.675930,1.640737,1.603491,1.564316,1.522561,1.480333,1.437632
2522,['H3N2'],[100000.0],0.071136,0.082153,0.091068,0.099373,0.106763,0.112949,0.118864,0.121186,...,1.734593,1.702339,1.668220,1.632780,1.595763,1.557542,1.518068,1.477373,1.435153,1.391475
2533,['HMPVA'],[100000.0],0.040729,0.044136,0.048949,0.054610,0.060898,0.067644,0.076390,0.081068,...,0.961932,0.952593,0.941746,0.929186,0.915051,0.899339,0.882271,0.863085,0.844153,0.825271


In [34]:
# Prepare reference data
reference_series = pd.read_csv(DATA_FOLDER + "/SiO2 - only GLF.txt", sep='\t', header=None, index_col=0).squeeze('columns')
reference_series.index = reference_series.index.astype(str) + '.0'  # Convert index to string and append '.0'

# Ensure reference series has the same wavenumbers
wavenumbers = highest_single_virus_conc_df.columns[2:]  # Exclude 'Label' and 'Conc'
reference_series = reference_series.reindex(wavenumbers)

In [35]:
# Step 5: Process the fixed dataframe
print("\nProcessing all samples...")
predicted_df, result_df = process_dataframe(fixed_df, parsed_data, highest_single_virus_conc_df, reference_series)


Processing all samples...

Processing sample: Label=['Ad5'], Conc=[100.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [100.0]
Fitted coefficients: [0.20600991 0.79399009]

Processing sample: Label=['Ad5'], Conc=[100000.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [100000.0]
Fitted coefficients: [9.99999925e-01 7.52312299e-08]

Processing sample: Label=['Ad5'], Conc=[12500.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [12500.0]
Fitted coefficients: [0.81237653 0.18762347]

Processing sample: Label=['Ad5'], Conc=[1562.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [1562.0]
Fitted coefficients: [0.59467719 0.40532281]

Processing sample: Label=['Ad5'], Conc=[195.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [195.0]
Fitted coefficients: [0.27403126 0.72596874]

Processing sample: Label=['Ad5'], Conc=[25000.0]
Parsed viruses: ['Ad5']
Parsed concentrations: [25000.0]
Fitted coefficients: [0.87731693 0.12268307]

Processing sample: Label=['Ad5'], Conc=[3125.0]
Parsed virus

/tmp/ipykernel_2529799/2434598198.py:386: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat([result_df, pd.DataFrame([result_row])], ignore_index=True)



Processing sample: Label=['CoV229E'], Conc=[12500.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [12500.0]
Fitted coefficients: [0.89482784 0.10517216]

Processing sample: Label=['CoV229E'], Conc=[1562.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [1562.0]
Fitted coefficients: [0.8582144 0.1417856]

Processing sample: Label=['CoV229E'], Conc=[195.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [195.0]
Fitted coefficients: [0.40702021 0.59297979]

Processing sample: Label=['CoV229E'], Conc=[25000.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [25000.0]
Fitted coefficients: [0.95944541 0.04055459]

Processing sample: Label=['CoV229E'], Conc=[3125.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [3125.0]
Fitted coefficients: [0.86966512 0.13033488]

Processing sample: Label=['CoV229E'], Conc=[391.0]
Parsed viruses: ['CoV229E']
Parsed concentrations: [391.0]
Fitted coefficients: [0.60583677 0.39416323]

Processing sample: Label=['CoV229E'], Conc=[50.0]
Par

/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:441: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:495: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)



Processing sample: Label=['CoVNL63', 'FluB'], Conc=[100000.0, 195.0]
Parsed viruses: ['CoVNL63', 'FluB']
Parsed concentrations: [100000.0, 195.0]
Fitted coefficients: [0.57698449 0.02584177 0.39717375]

Processing sample: Label=['CoVNL63', 'FluB'], Conc=[100000.0, 25000.0]
Parsed viruses: ['CoVNL63', 'FluB']
Parsed concentrations: [100000.0, 25000.0]
Fitted coefficients: [0.53896246 0.38492333 0.07611421]

Processing sample: Label=['CoVNL63', 'FluB'], Conc=[100000.0, 3125.0]
Parsed viruses: ['CoVNL63', 'FluB']
Parsed concentrations: [100000.0, 3125.0]
Fitted coefficients: [0.55747317 0.23557572 0.2069511 ]

Processing sample: Label=['CoVNL63', 'FluB'], Conc=[100000.0, 391.0]
Parsed viruses: ['CoVNL63', 'FluB']
Parsed concentrations: [100000.0, 391.0]
Fitted coefficients: [0.59161197 0.05975729 0.34863074]

Processing sample: Label=['CoVNL63', 'FluB'], Conc=[100000.0, 50000.0]
Parsed viruses: ['CoVNL63', 'FluB']
Parsed concentrations: [100000.0, 50000.0]
Fitted coefficients: [0.5548040

...


Processing sample: Label=['H3N2', 'RSVB1'], Conc=[3125.0, 50000.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [3125.0, 50000.0]
Fitted coefficients: [0.08041577 0.67585726 0.24372697]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[3125.0, 6250.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [3125.0, 6250.0]
Fitted coefficients: [0.07660233 0.56925716 0.35414051]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[3125.0, 781.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [3125.0, 781.0]
Fitted coefficients: [0.06910004 0.39969706 0.5312029 ]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[391.0, 100000.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [391.0, 100000.0]
Fitted coefficients: [0.01478334 0.80278758 0.18242908]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[391.0, 12500.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [391.0, 12500.0]
Fitted coefficients: [0.01378234 0.63283849 0.35337917]

Processing s

/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)



Processing sample: Label=['H3N2', 'RSVB1'], Conc=[50000.0, 3125.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [50000.0, 3125.0]
Fitted coefficients: [0.54068057 0.12175086 0.33756857]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[50000.0, 391.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [50000.0, 391.0]
Fitted coefficients: [0.52621551 0.00525163 0.46853287]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[50000.0, 50000.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [50000.0, 50000.0]
Fitted coefficients: [0.51952511 0.26322123 0.21725366]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[50000.0, 6250.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [50000.0, 6250.0]
Fitted coefficients: [0.54044186 0.17579042 0.28376772]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[50000.0, 781.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [50000.0, 781.0]
Fitted coefficients: [0.53916792 0.07771654 0.38311554]

Proces

/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/home/zhao/myenv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)



Processing sample: Label=['H3N2', 'RSVB1'], Conc=[781.0, 1562.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [781.0, 1562.0]
Fitted coefficients: [0.02085101 0.45962749 0.5195215 ]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[781.0, 195.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [781.0, 195.0]
Fitted coefficients: [0.04356186 0.00212693 0.95431122]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[781.0, 25000.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [781.0, 25000.0]
Fitted coefficients: [0.03422019 0.64982559 0.31595422]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[781.0, 3125.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [781.0, 3125.0]
Fitted coefficients: [0.01864728 0.48575989 0.49559284]

Processing sample: Label=['H3N2', 'RSVB1'], Conc=[781.0, 391.0]
Parsed viruses: ['H3N2', 'RSVB1']
Parsed concentrations: [781.0, 391.0]
Fitted coefficients: [0.01090873 0.25230584 0.73678543]

Processing sample: Label=[

In [36]:
# Step 6: Sort the wavenumber columns if needed
if not predicted_df.empty:
    wavenumber_cols = [col for col in predicted_df.columns if col not in ['Label', 'Conc']]
    wavenumber_cols_sorted = sorted(wavenumber_cols, key=float)
    desired_order_cols = ['Label', 'Conc'] + wavenumber_cols_sorted
    predicted_df = predicted_df[desired_order_cols]

predicted_df

,Label,Conc,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,...,1691.0,1692.0,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0
0,['Ad5'],[100.0],0.146968,0.153877,0.161078,0.168916,0.177164,0.185491,0.195904,0.201364,...,0.797566,0.781133,0.765924,0.751674,0.739240,0.725515,0.710731,0.693833,0.677220,0.660722
1,['Ad5'],[100000.0],0.108017,0.115627,0.123576,0.132051,0.140915,0.149695,0.160068,0.166644,...,1.058237,1.045763,1.033407,1.020407,1.007390,0.992627,0.976119,0.957441,0.937000,0.914610
2,['Ad5'],[12500.0],0.117221,0.124666,0.132438,0.140762,0.149481,0.158154,0.168536,0.174848,...,0.996639,0.983229,0.970199,0.956904,0.944025,0.929507,0.913406,0.895149,0.875613,0.854615
3,['Ad5'],[1562.0],0.127901,0.135153,0.142721,0.150870,0.159420,0.167968,0.178362,0.184368,...,0.925168,0.910672,0.896860,0.883222,0.870503,0.856270,0.840641,0.822872,0.804385,0.785003
4,['Ad5'],[195.0],0.143631,0.150600,0.157866,0.165757,0.174058,0.182424,0.192834,0.198389,...,0.819898,0.803804,0.788840,0.774696,0.762213,0.748399,0.733467,0.716416,0.699475,0.682473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,['RSVB1'],[391.0],0.159109,0.165067,0.170919,0.177092,0.183435,0.189692,0.197186,0.201110,...,0.794861,0.779894,0.766236,0.753827,0.743455,0.731863,0.719226,0.704624,0.690286,0.676152
2575,['RSVB1'],[50.0],0.157547,0.164095,0.170834,0.178158,0.185841,0.193597,0.203340,0.208221,...,0.745012,0.728130,0.712714,0.698642,0.686804,0.673781,0.659838,0.643829,0.628506,0.613731
2576,['RSVB1'],[50000.0],0.162649,0.167270,0.171110,0.174676,0.177982,0.180843,0.183237,0.184993,...,0.907845,0.897215,0.887544,0.878905,0.871857,0.863505,0.853827,0.842418,0.830310,0.817629
2577,['RSVB1'],[6250.0],0.161704,0.166682,0.171059,0.175321,0.179437,0.183205,0.186960,0.189294,...,0.877689,0.865901,0.855166,0.845521,0.837586,0.828369,0.817901,0.805640,0.792936,0.779868


In [37]:
result_df

,Label,Conc,Coefficients,Intercept,MAE,RMSE,R2,VIFs
0,['Ad5'],[100.0],"[0.20600990733159996, 0.7939900911335066]",0,5.040000e-02,6.268144e-02,0.988069,"[1.2393174134508542, 1.2393174134508544]"
1,['Ad5'],[100000.0],"[0.9999999247687703, 7.523122991237585e-08]",0,3.838299e-08,4.822219e-08,1.000000,"[1.2393174134508542, 1.2393174134508544]"
2,['Ad5'],[12500.0],"[0.812376534461888, 0.18762346561883003]",0,5.025815e-02,6.322162e-02,0.975363,"[1.2393174134508542, 1.2393174134508544]"
3,['Ad5'],[1562.0],"[0.5946771859918338, 0.40532281098829875]",0,1.055268e-01,1.332815e-01,0.913586,"[1.2393174134508542, 1.2393174134508544]"
4,['Ad5'],[195.0],"[0.27403126111066306, 0.725968738889337]",0,9.737364e-02,1.238858e-01,0.954032,"[1.2393174134508542, 1.2393174134508544]"
...,...,...,...,...,...,...,...,...
2574,['RSVB1'],[391.0],"[0.31958347302586476, 0.6804165269741352]",0,8.454475e-02,1.081358e-01,0.976600,"[2.604941074035777, 2.6049410740357786]"
2575,['RSVB1'],[50.0],"[0.07422413634395475, 0.9257758636560453]",0,3.616650e-02,5.415651e-02,0.993632,"[2.604941074035777, 2.6049410740357786]"
2576,['RSVB1'],[50000.0],"[0.8756893240199906, 0.12431067606757835]",0,5.275504e-02,6.881450e-02,0.988676,"[2.604941074035777, 2.6049410740357786]"
2577,['RSVB1'],[6250.0],"[0.7272606699824804, 0.27273932798545336]",0,1.031624e-01,1.427801e-01,0.950685,"[2.604941074035777, 2.6049410740357786]"


In [38]:
# Step 7: Save the processed data
print("\nSaving processed data...")
pd.DataFrame(predicted_df).to_csv(RESULT_FOLDER + f"/{DATE} - reconstructed_spectra_(only_GLF)_constraint_1e-10.csv", index=False)
pd.DataFrame(result_df).to_csv(RESULT_FOLDER + f"/{DATE} - coefficients_(only_GLF)_constraint_1e-10.csv", index=False)


Saving processed data...


In [39]:
# Step 8: Calculate group average of predicted data
print("\nCalculating group average of predicted data...")
avg_predicted_df = calculate_group_average(predicted_df)
print(f"Average predicted DataFrame shape: {avg_predicted_df.shape}")
avg_predicted_df


Calculating group average of predicted data...
Average predicted DataFrame shape: (2579, 1251)


450.0     451.0     452.0     453.0     454.0  \
Label     Conc                                                           
['Ad5']   [100.0]     0.146968  0.153877  0.161078  0.168916  0.177164   
          [100000.0]  0.108017  0.115627  0.123576  0.132051  0.140915   
          [12500.0]   0.117221  0.124666  0.132438  0.140762  0.149481   
          [1562.0]    0.127901  0.135153  0.142721  0.150870  0.159420   
          [195.0]     0.143631  0.150600  0.157866  0.165757  0.174058   
...                        ...       ...       ...       ...       ...   
['RSVB1'] [391.0]     0.159109  0.165067  0.170919  0.177092  0.183435   
          [50.0]      0.157547  0.164095  0.170834  0.178158  0.185841   
          [50000.0]   0.162649  0.167270  0.171110  0.174676  0.177982   
          [6250.0]    0.161704  0.166682  0.171059  0.175321  0.179437   
          [781.0]     0.160274  0.165792  0.170982  0.176297  0.181640   

                         455.0     456.0     457.0     458.0     459.0  ...  \
Label     Conc                                                          ...   
['Ad5']   [100.0]     0.185491  0.195904  0.201364  0.205159  0.207942  ...   
          [100000.0]  0.149695  0.160068  0.166644  0.171949  0.176271  ...   
          [12500.0]   0.158154  0.168536  0.174848  0.179797  0.183755  ...   
          [1562.0]    0.167968  0.178362  0.184368  0.188902  0.192439  ...   
          [195.0]     0.182424  0.192834  0.198389  0.202314  0.205228  ...   
...                        ...       ...       ...       ...       ...  ...   
['RSVB1'] [391.0]     0.189692  0.197186  0.201110  0.203707  0.205627  ...   
          [50.0]      0.193597  0.203340  0.208221  0.211437  0.213713  ...   
          [50000.0]   0.180843  0.183237  0.184993  0.186188  0.187300  ...   
          [6250.0]    0.183205  0.186960  0.189294  0.190864  0.192192  ...   
          [781.0]     0.186779  0.192593  0.195804  0.197940  0.199593  ...   

                        1691.0    1692.0    1693.0    1694.0    1695.0  \
Label     Conc                                                           
['Ad5']   [100.0]     0.797566  0.781133  0.765924  0.751674  0.739240   
          [100000.0]  1.058237  1.045763  1.033407  1.020407  1.007390   
          [12500.0]   0.996639  0.983229  0.970199  0.956904  0.944025   
          [1562.0]    0.925168  0.910672  0.896860  0.883222  0.870503   
          [195.0]     0.819898  0.803804  0.788840  0.774696  0.762213   
...                        ...       ...       ...       ...       ...   
['RSVB1'] [391.0]     0.794861  0.779894  0.766236  0.753827  0.743455   
          [50.0]      0.745012  0.728130  0.712714  0.698642  0.686804   
          [50000.0]   0.907845  0.897215  0.887544  0.878905  0.871857   
          [6250.0]    0.877689  0.865901  0.855166  0.845521  0.837586   
          [781.0]     0.832057  0.818517  0.806172  0.795005  0.785727   

                        1696.0    1697.0    1698.0    1699.0    1700.0  
Label     Conc                                                          
['Ad5']   [100.0]     0.725515  0.710731  0.693833  0.677220  0.660722  
          [100000.0]  0.992627  0.976119  0.957441  0.937000  0.914610  
          [12500.0]   0.929507  0.913406  0.895149  0.875613  0.854615  
          [1562.0]    0.856270  0.840641  0.822872  0.804385  0.785003  
          [195.0]     0.748399  0.733467  0.716416  0.699475  0.682473  
...                        ...       ...       ...       ...       ...  
['RSVB1'] [391.0]     0.731863  0.719226  0.704624  0.690286  0.676152  
          [50.0]      0.673781  0.659838  0.643829  0.628506  0.613731  
          [50000.0]   0.863505  0.853827  0.842418  0.830310  0.817629  
          [6250.0]    0.828369  0.817901  0.805640  0.792936  0.779868  
          [781.0]     0.775201  0.763538  0.749988  0.736383  0.722728  

[2579 rows x 1251 columns]

In [40]:
# Step 9: Get highest concentration rows from predicted data
highest_single_virus_conc_predicted_df = get_single_virus_highest_concentration_rows(avg_predicted_df)
highest_single_virus_conc_predicted_df

Extracting single virus data...
Found 149 single virus samples
Example concentration values:
['[100.0]', '[100000.0]', '[12500.0]', '[1562.0]', '[195.0]']
After parsing concentrations:
     Label        Conc  Conc_parsed
0  ['Ad5']     [100.0]        100.0
1  ['Ad5']  [100000.0]     100000.0
2  ['Ad5']   [12500.0]      12500.0
3  ['Ad5']    [1562.0]       1562.0
4  ['Ad5']     [195.0]        195.0
Found 13 unique single virus types
Highest concentration for ['Ad5']: [100000.0]
Highest concentration for ['CoV2']: [100000.0]
Highest concentration for ['CoV229E']: [100000.0]
Highest concentration for ['CoV2B1']: [100000.0]
Highest concentration for ['CoVNL63']: [100000.0]
Highest concentration for ['CoVOC43']: [100000.0]
Highest concentration for ['FluB']: [100000.0]
Highest concentration for ['H1N1']: [100000.0]
Highest concentration for ['H3N2']: [100000.0]
Highest concentration for ['HMPVA']: [100000.0]
Highest concentration for ['HMPVB']: [100000.0]
Highest concentration for ['RSVA2']

,Label,Conc,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,...,1691.0,1692.0,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0
1,['Ad5'],[100000.0],0.108017,0.115627,0.123576,0.132051,0.140915,0.149695,0.160068,0.166644,...,1.058237,1.045763,1.033407,1.020407,1.007390,0.992627,0.976119,0.957441,0.937000,0.914610
12,['CoV2'],[100000.0],0.143678,0.158237,0.170542,0.182068,0.192441,0.201322,0.209508,0.215186,...,0.970017,0.953678,0.940966,0.930339,0.924712,0.915695,0.904000,0.887271,0.871712,0.857068
23,['CoV229E'],[100000.0],0.026228,0.023702,0.023737,0.024930,0.027228,0.030421,0.036719,0.038281,...,1.206211,1.187544,1.167246,1.145368,1.122035,1.097088,1.070860,1.042912,1.014982,0.987281
35,['CoV2B1'],[100000.0],0.309544,0.322877,0.337579,0.353386,0.369930,0.386789,0.407053,0.419105,...,1.195842,1.181596,1.163491,1.141491,1.114737,1.087912,1.061368,1.034211,1.010965,0.990649
2036,['CoVNL63'],[100000.0],0.059492,0.065085,0.071356,0.077983,0.084492,0.090576,0.098661,0.100508,...,1.009051,1.005831,0.999797,0.991441,0.979644,0.967000,0.953153,0.938627,0.922085,0.903356
2048,['CoVOC43'],[100000.0],0.131847,0.137356,0.146068,0.156712,0.168932,0.182966,0.200000,0.214000,...,1.144441,1.152814,1.158136,1.161000,1.160203,1.157237,1.151763,1.143576,1.132695,1.118966
2060,['FluB'],[100000.0],0.109414,0.115172,0.122069,0.129431,0.137362,0.145586,0.154707,0.162293,...,1.108776,1.088431,1.070414,1.053431,1.040483,1.022914,1.001603,0.973517,0.946621,0.920534
2291,['H1N1'],[100000.0],0.055175,0.060404,0.066579,0.073368,0.080491,0.087807,0.096965,0.102035,...,1.770947,1.741158,1.709351,1.675930,1.640737,1.603491,1.564316,1.522561,1.480333,1.437632
2522,['H3N2'],[100000.0],0.071136,0.082153,0.091068,0.099373,0.106763,0.112949,0.118864,0.121186,...,1.734593,1.702339,1.668220,1.632780,1.595763,1.557542,1.518068,1.477373,1.435153,1.391475
2533,['HMPVA'],[100000.0],0.040729,0.044136,0.048949,0.054610,0.060898,0.067644,0.076390,0.081068,...,0.961932,0.952593,0.941746,0.929186,0.915051,0.899339,0.882271,0.863085,0.844153,0.825271


In [41]:
# Step 10: Save highest concentration predicted data
pd.DataFrame(highest_single_virus_conc_predicted_df).to_csv(
    RESULT_FOLDER + f"/{DATE} - reconstructed_spectra_highest_concentration_(only_GLF)_constraint_1e-10.csv", 
    index=False
)

# Pred vs Real spectra

In [42]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import ast
import seaborn as sns

In [43]:
# Set up folders
DIFF_FOLDER = f"{WORKING_PATH}/{DATE} - difference_analysis"
R2_MAE_FOLDER = f"{WORKING_PATH}/{DATE} - r2_mae_analysis"

# Make sure folders exist
os.makedirs(DIFF_FOLDER, exist_ok=True)
os.makedirs(R2_MAE_FOLDER, exist_ok=True)

In [44]:
# 1. Basic font configuration
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# 2. Function to explicitly set font sizes and family on all text elements
def set_explicit_text_properties(fig, fontsize=None):
    """
    Sets font properties on all text elements in the figure.
    """
    for ax in fig.axes:
        # Set title font
        if ax.get_title():
            ax.set_title(ax.get_title(), fontname='Times New Roman', fontsize=28 if fontsize is None else fontsize)
        
        # Set axis label fonts
        ax.set_xlabel(ax.get_xlabel(), fontname='Times New Roman', fontsize=32 if fontsize is None else fontsize)
        ax.set_ylabel(ax.get_ylabel(), fontname='Times New Roman', fontsize=32 if fontsize is None else fontsize)
        
        # Set tick label fonts
        for tick in ax.get_xticklabels():
            tick.set_fontname('Times New Roman')
            tick.set_fontsize(28 if fontsize is None else fontsize)
        
        for tick in ax.get_yticklabels():
            tick.set_fontname('Times New Roman')
            tick.set_fontsize(28 if fontsize is None else fontsize)
        
        # Set legend font
        if ax.get_legend():
            for text in ax.get_legend().get_texts():
                text.set_fontname('Times New Roman')
                text.set_fontsize(20 if fontsize is None else fontsize)
    
    return fig

In [45]:
# Function to format concentration as integer if possible
def format_concentration(conc):
    try:
        # Check if it can be converted to an integer without loss
        if float(conc) == int(float(conc)):
            return str(int(float(conc)))
        return str(conc)
    except:
        return str(conc)

# Function to fix original data format for mixed viruses
def fix_original_data(original_df):
    """
    Fix mixed virus entries where part of the concentration is in the label
    """
    print("\nFixing the mixed virus format in the original data...")
    
    # Function to fix a mixed virus label and concentration
    def fix_mixed_virus_entry(label, conc):
        """
        Fix mixed virus entries where part of the concentration is in the label
        
        Args:
            label: The possibly corrupted label
            conc: The current concentration value
            
        Returns:
            tuple: (fixed_label, fixed_conc)
        """
        # Check if this is a corrupted mixed virus entry
        if isinstance(label, str) and '_[' in label:
            # Split at the '_[' which separates the virus names from the concentration
            parts = label.split('_[')
            if len(parts) >= 2:
                # First part is the fixed label
                fixed_label = parts[0]
                
                # Second part combined with conc is the fixed concentration
                fixed_conc = f"[{parts[1]}{conc}".rstrip("]") + "]"
                
                # If multiple '_[' sequences, handle more complex cases
                if len(parts) > 2:
                    # More complex parsing might be needed
                    pass
                    
                return fixed_label, fixed_conc
        
        # If not a corrupted entry or parsing fails, return original values
        return label, conc
    
    # Create a new dataframe with fixed labels and concentrations
    original_df_fixed = original_df.copy()
    
    # Apply the fix to each row
    fixed_rows = []
    for idx, row in original_df.iterrows():
        # Make a copy of the row
        new_row = row.copy()
        
        # Fix the label and concentration
        fixed_label, fixed_conc = fix_mixed_virus_entry(row['Label'], row['Conc'])
        
        # Update the row with fixed values
        new_row['Label'] = fixed_label
        new_row['Conc'] = fixed_conc
        
        # Add to the list of fixed rows
        fixed_rows.append(new_row)
    
    # Create a new dataframe from the fixed rows
    original_df_fixed = pd.DataFrame(fixed_rows)
    
    # Check that the fixing worked
    print("\nChecking fixed data:")
    mixed_virus_count = 0
    for idx, row in original_df_fixed.iterrows():
        label = row['Label']
        conc = row['Conc']
        
        if isinstance(label, str) and "'__'" in str(label) and '_[' not in str(label):
            mixed_virus_count += 1
            
            if mixed_virus_count <= 5:  # Show first 5 examples
                print(f"Fixed mixed virus: Label={label}, Conc={conc}")
    
    print(f"\nFound {mixed_virus_count} properly formatted mixed virus entries in the fixed data")
    
    return original_df_fixed

# Function to find matching concentration in another DataFrame
def find_matching_conc(label, conc_str, df):
    """
    Find a matching concentration for the given label in the DataFrame,
    using flexible matching to handle different string representations.
    """
    # Get all concentrations for this label in the DataFrame
    if not isinstance(df, pd.DataFrame) or df.empty:
        return None
        
    label_rows = df[df['Label'] == label]
    if label_rows.empty:
        return None
        
    # Try exact match
    if conc_str in label_rows['Conc'].values:
        return conc_str
        
    # Try comparing the actual values
    try:
        # Parse input concentration
        if isinstance(conc_str, str) and conc_str.startswith('[') and conc_str.endswith(']'):
            input_values = ast.literal_eval(conc_str)
            
            # Compare with each concentration in the DataFrame
            for df_conc in label_rows['Conc'].values:
                if isinstance(df_conc, str) and df_conc.startswith('[') and df_conc.endswith(']'):
                    try:
                        df_values = ast.literal_eval(df_conc)
                        # Check if values match, ignoring the string format
                        if input_values == df_values:
                            return df_conc
                    except:
                        continue
    except:
        pass
        
    # No match found
    return None

# Function to calculate wavelength regions with large differences
def find_difference_regions(real_spectrum, predicted_spectrum, wavenumbers, threshold_percentage=10):
    # Calculate absolute differences
    differences = np.abs(predicted_spectrum - real_spectrum)
    
    # Find the max difference value
    max_diff = np.max(differences)
    
    # Set a threshold as a percentage of the maximum difference
    threshold = max_diff * (threshold_percentage/100)
    
    # Find regions where difference exceeds threshold
    significant_regions = []
    in_region = False
    start_idx = None
    
    for i, diff in enumerate(differences):
        if diff > threshold and not in_region:
            in_region = True
            start_idx = i
        elif diff <= threshold and in_region:
            in_region = False
            significant_regions.append((wavenumbers[start_idx], wavenumbers[i-1]))
    
    # Handle case where the last region extends to the end
    if in_region:
        significant_regions.append((wavenumbers[start_idx], wavenumbers[-1]))
    
    return significant_regions, differences

# Create lookup dictionary for the real data for efficient matching
def create_lookup_dictionary(df):
    """
    Create a lookup dictionary for a DataFrame for efficient flexible matching.
    """
    lookup_dict = {}
    
    # For each label and concentration, store the actual data row index for fast lookup
    for idx, row in df.reset_index().iterrows():
        if 'Label' not in row or 'Conc' not in row:
            continue
            
        label = row['Label']
        conc = row['Conc']
        
        # Skip if label or conc is missing
        if pd.isna(label) or pd.isna(conc):
            continue
        
        # Store this row's index
        if label not in lookup_dict:
            lookup_dict[label] = {}
        
        # Store the original concentration
        lookup_dict[label][conc] = idx
        
        # Also store normalized versions of the concentration
        if isinstance(conc, str):
            # Store without brackets
            clean_conc = conc.strip('[]')
            lookup_dict[label][clean_conc] = idx
            
            # Store with double brackets (common parsing artifact)
            lookup_dict[label][f"[{conc}]"] = idx
            
            # For concentrations with underscores, store each possible format
            if '_' in conc:
                # Handle [100000.0_100000.0] format
                if conc.startswith('[') and conc.endswith(']'):
                    values = conc.strip('[]').split('_')
                    
                    # Store as list format
                    list_format = str(values)
                    lookup_dict[label][list_format] = idx
                    
                    # Store as tuple format
                    tuple_format = str(tuple(values))
                    lookup_dict[label][tuple_format] = idx
    
    return lookup_dict

# Fix and prepare data function - modified to handle MultiIndex consistently
def fix_and_prepare_data():
    """
    Fix any anomalies in the data and prepare lookup dictionaries for efficient matching.
    Ensures the returned DataFrame has the expected format for later use.
    """
    print("\nFixing data anomalies and preparing lookup dictionaries...")
    
    # First check if we need to fix the original data
    needs_fixing = False
    
    # Find all mixed virus entries in the original data
    mixed_virus_count = 0
    for idx, row in df.iterrows():
        if isinstance(row['Label'], str) and '_[' in row['Label']:
            mixed_virus_count += 1
            needs_fixing = True
            if mixed_virus_count <= 3:  # Show just a few examples
                print(f"Found corrupted mixed virus entry: {row['Label']}, {row['Conc']}")
    
    # If we found corrupted entries, fix the original data
    if needs_fixing:
        print(f"Found {mixed_virus_count} corrupted mixed virus entries, fixing...")
        fixed_original_df = fix_original_data(df)
        
        # Create new grouped data
        new_grouped_df = fixed_original_df.groupby(['Label', 'Conc']).mean()
        
        # Reset index to make Label and Conc columns for easier filtering
        new_grouped_df_reset = new_grouped_df.reset_index()
        
        # Create lookup dictionary from the original MultiIndex form
        real_lookup_dict = create_lookup_dictionary(new_grouped_df)
        
        return new_grouped_df_reset, real_lookup_dict
    else:
        print("No data anomalies found, using existing data.")
        
        # Check if grouped_df is a MultiIndex DataFrame
        if isinstance(grouped_df.index, pd.MultiIndex) and 'Label' in grouped_df.index.names:
            # Reset index to make Label and Conc columns for easier filtering
            grouped_df_reset = grouped_df.reset_index()
            real_lookup_dict = create_lookup_dictionary(grouped_df)
            return grouped_df_reset, real_lookup_dict
        else:
            # Already in the right format
            real_lookup_dict = create_lookup_dictionary(grouped_df)
            return grouped_df, real_lookup_dict

In [54]:
# Function to create standard comparison plots (real vs predicted)
def create_comparison_plot(label, conc, predicted_df, grouped_df_fixed, real_lookup_dict=None):
    """
    Create standard comparison plots showing real vs predicted spectra.
    """
    print(f"\nCreating comparison plot for {label} at {conc}")
    
    # Get predicted data
    pred_data = predicted_df[(predicted_df['Label'] == label) & (predicted_df['Conc'] == conc)]
    if pred_data.empty:
        print(f"  No predicted data found for {label} at {conc}")
        return
    
    # Get real data if available
    real_data = None
    comparison_possible = False
    
    if isinstance(grouped_df_fixed, pd.DataFrame):
        # First try direct match
        real_data = grouped_df_fixed[(grouped_df_fixed['Label'] == label) & (grouped_df_fixed['Conc'] == conc)]
        
        # If no direct match, try using the lookup dictionary
        if real_data.empty and real_lookup_dict is not None and label in real_lookup_dict:
            for alt_conc in real_lookup_dict.get(label, {}):
                idx = real_lookup_dict[label][alt_conc]
                real_data = grouped_df_fixed.iloc[[idx]]
                if not real_data.empty:
                    print(f"  Found real data using lookup dictionary with concentration: {alt_conc}")
                    break
        
        if not real_data.empty:
            comparison_possible = True
    
    # Extract wavenumbers
    wavenumbers = pred_data.columns[2:].astype(float)
    
    # Create figure for the plot
    fig = plt.figure(figsize=(8.7, 8))
    plt.xlabel('Wavenumbers')
    plt.ylabel('Intensity')
    
    # Plot real spectrum first (blue, solid)
    if comparison_possible and real_data is not None:
        real_spectrum = real_data.iloc[0, 2:].values
        plt.plot(wavenumbers, real_spectrum, color='blue', linewidth=2, linestyle='-', label='Real Spectrum')
    
    # Plot predicted spectrum (orange, dashed)
    predicted_spectrum = pred_data.iloc[0, 2:].values
    plt.plot(wavenumbers, predicted_spectrum, color='orange', linewidth=2, linestyle='--', label='Reconstructed Spectrum')
    
    # Calculate MAE if comparison is possible
    if comparison_possible and real_data is not None:
        mae = np.mean(np.abs(real_spectrum - predicted_spectrum))
        plt.title(f"MAE: {mae:.4f}")
        
        # Add to MAE records
        mae_records.append({
            'Label': label,
            'Concentration': conc,
            'MAE': mae
        })
    
    # Adjusting x-axis ticks
    tick_interval = 200.0
    plt.xticks(np.arange(start=450, stop=1701, step=tick_interval))
    
    # Add legend
    plt.legend(loc='best')
    set_explicit_text_properties(fig)
    plt.tight_layout()
    
    # Create a safe filename
    safe_label = str(label).replace('/', '_').replace('\\', '_').replace(':', '_')
    safe_conc = str(conc).replace('/', '_').replace('\\', '_').replace(':', '_')
    
    # Save the plot
    filename = f"{RESULT_FOLDER}/{DATE} - {safe_label}_Concentration_{safe_conc}.png"
    plt.savefig(filename, dpi=300)
    plt.show()
    plt.close('all')
    plt.clf()
    plt.cla()
    
    # Create difference plot if comparison is possible
    if comparison_possible and real_data is not None:
        fig_diff = plt.figure(figsize=(8.7, 8))
        plt.xlabel('Wavenumbers')
        plt.ylabel('Intensity Difference')
        
        difference = predicted_spectrum - real_spectrum
        plt.plot(wavenumbers, difference, color='purple', linewidth=2, label='Reconstructed - Real')
        
        # Add a zero line for reference
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        
        # Adjusting x-axis ticks
        plt.xticks(np.arange(start=450, stop=1701, step=tick_interval))
        plt.legend()
        set_explicit_text_properties(fig_diff)
        plt.tight_layout()
        
        # Filename for difference plot
        diff_filename = f"{RESULT_FOLDER}/{DATE} - {safe_label}_Concentration_{safe_conc}_difference.png"
        plt.savefig(diff_filename, dpi=300)
        plt.show()
        plt.close('all')
        plt.clf()
        plt.cla()

# Function to create enhanced plots for all samples
def create_enhanced_plot(label, conc, predicted_df, grouped_df, highest_single_virus_conc_df, reference_series, real_lookup_dict=None):
    """
    Create enhanced plots for ALL samples (single and mixed virus) that include:
    1. Predicted spectrum (orange, dashed)
    2. Real spectrum (blue, solid) - if available
    3. All component spectra at their respective highest concentrations
    4. SiO2 reference component
    
    Args:
        label: Virus label (e.g., "['Ad5']" or "['CoVNL63', 'FluB']")
        conc: Concentration string or value
        predicted_df: DataFrame with predicted data
        grouped_df: DataFrame with real data
        highest_single_virus_conc_df: DataFrame with highest concentration single virus data
        reference_series: Series with SiO2 reference data
        real_lookup_dict: Dictionary for flexible matching of real data (optional)
    """
    import ast
    print(f"\nCreating enhanced plot for {label} at {conc}")
    
    # Get predicted data
    pred_data = predicted_df[(predicted_df['Label'] == label) & (predicted_df['Conc'] == conc)]
    if pred_data.empty:
        print(f"  No predicted data found for {label} at {conc}")
        return
    
    # Get real data if available
    real_data = None
    comparison_possible = False
    
    if isinstance(grouped_df, pd.DataFrame):
        # First try direct match
        real_data = grouped_df[(grouped_df['Label'] == label) & (grouped_df['Conc'] == conc)]
        
        # If no direct match, try using the lookup dictionary
        if real_data.empty and real_lookup_dict is not None and label in real_lookup_dict:
            for alt_conc in real_lookup_dict.get(label, {}):
                idx = real_lookup_dict[label][alt_conc]
                real_data = grouped_df.iloc[[idx]]
                if not real_data.empty:
                    print(f"  Found real data using lookup dictionary with concentration: {alt_conc}")
                    break
        
        if not real_data.empty:
            comparison_possible = True
    
    # Extract wavenumbers
    wavenumbers = pred_data.columns[2:].astype(float)
    
    # Parse component viruses from the label
    component_viruses = []
    
    try:
        # Try to use ast.literal_eval first to parse the label
        if isinstance(label, str) and label.startswith('[') and label.endswith(']'):
            try:
                parsed_label = ast.literal_eval(label)
                if isinstance(parsed_label, list):
                    component_viruses = parsed_label
                else:
                    component_viruses = [str(parsed_label)]
            except (SyntaxError, ValueError):
                # If that fails, use string splitting
                if "'__'" in label:
                    # Mixed virus case
                    clean_label = label.strip("[]' ")
                    parts = clean_label.split("'__'")
                    component_viruses = [part.strip("'") for part in parts]
                else:
                    # Single virus case
                    clean_label = label.strip("[]' ")
                    component_viruses = [clean_label]
        else:
            # Non-standard format, just use the label as is
            component_viruses = [label]
    except Exception as e:
        print(f"  Error parsing virus label: {e}")
        component_viruses = [str(label)]  # Fallback
    
    print(f"  Component viruses: {component_viruses}")
    
    # Create figure for the plot
    fig = plt.figure(figsize=(12, 10))  # Larger figure for more elements
    plt.xlabel('Wavenumbers')
    plt.ylabel('Intensity')
    
    # Plot component virus spectra
    component_spectra = []
    legend_labels = []
    
    # Add each virus component spectrum
    for i, virus in enumerate(component_viruses):
        # Format the virus label for lookup
        virus_label = f"['{virus}']"
        
        # Find the virus in the highest concentration data
        if virus_label in highest_single_virus_conc_df['Label'].values:
            # Get data for this virus
            virus_data = highest_single_virus_conc_df[highest_single_virus_conc_df['Label'] == virus_label]
            
            # Get the spectrum
            if not virus_data.empty:
                virus_spectrum = virus_data.iloc[0, 2:].values
                component_spectra.append(virus_spectrum)
                
                # Get the highest concentration for this virus
                virus_conc = virus_data.iloc[0]['Conc']
                
                # Plot with a distinctive color and line style
                colors = ['green', 'purple', 'red', 'brown', 'cyan']  # Colors for components
                line_style = '-.'  # Different line style for components
                color = colors[i % len(colors)]  # Cycle through colors
                
                plt.plot(wavenumbers, virus_spectrum, color=color, linestyle=line_style, 
                         linewidth=1.5, alpha=0.7)
                
                # Add to legend labels - include the actual highest concentration
                legend_label = f"{virus} Component (Highest conc: {virus_conc})"
                legend_labels.append(legend_label)
                print(f"  Added component: {legend_label}")
    
    # Add SiO2 reference component
    plt.plot(wavenumbers, reference_series.values, color='dimgray', linestyle='-.', 
             linewidth=1.5, alpha=0.8)
    legend_labels.append("SiO2 Reference")
    print("  Added component: SiO2 Reference")
    
    # Plot real spectrum first (blue, solid)
    if comparison_possible and real_data is not None:
        real_spectrum = real_data.iloc[0, 2:].values
        plt.plot(wavenumbers, real_spectrum, color='blue', linewidth=2, linestyle='-', label='Real Spectrum')
    
    # Plot predicted spectrum (orange, dashed)
    predicted_spectrum = pred_data.iloc[0, 2:].values
    plt.plot(wavenumbers, predicted_spectrum, color='orange', linewidth=2, linestyle='--', label='Reconstructed Spectrum')
    
    # Calculate MAE if comparison is possible
    if comparison_possible and real_data is not None:
        mae = np.mean(np.abs(real_spectrum - predicted_spectrum))
        # plt.title(f"MAE: {mae:.4f}")
        
        # Add to MAE records if not already added by the comparison plot
        mae_record_exists = False
        for record in mae_records:
            if record['Label'] == label and record['Concentration'] == conc:
                mae_record_exists = True
                break
                
        if not mae_record_exists:
            mae_records.append({
                'Label': label,
                'Concentration': conc,
                'MAE': mae
            })
    
    # Add component legend labels
    for i, label_text in enumerate(legend_labels):
        if "SiO2" in label_text:
            color = 'gray'
            line_style = ':'
        else:
            colors = ['green', 'purple', 'red', 'brown', 'cyan']
            color = colors[i % len(colors) if i < len(colors) else 0]
            line_style = '-.'
        plt.plot([], [], color=color, linestyle=line_style, linewidth=1.5, alpha=0.7, label=label_text)
    
    # Adjusting x-axis ticks
    tick_interval = 200.0
    plt.xticks(np.arange(start=450, stop=1701, step=tick_interval))
    
    # Add legend
    plt.legend(loc='best')
    set_explicit_text_properties(fig)
    plt.tight_layout()
    
    # Create a safe filename
    safe_label = str(label).replace('/', '_').replace('\\', '_').replace(':', '_')
    safe_conc = str(conc).replace('/', '_').replace('\\', '_').replace(':', '_')
    
    # Save the plot
    filename = f"{RESULT_FOLDER}/{DATE} - {safe_label}_Concentration_{safe_conc}_enhanced.png"
    plt.savefig(filename, dpi=300)
    plt.show()
    plt.close('all')
    plt.clf()
    plt.cla()
    
    # Create difference plot if comparison is possible
    if comparison_possible and real_data is not None:
        fig_diff = plt.figure(figsize=(10, 8))
        plt.xlabel('Wavenumbers')
        plt.ylabel('Intensity Difference')
        
        difference = predicted_spectrum - real_spectrum
        plt.plot(wavenumbers, difference, color='purple', linewidth=2, label='Reconstructed - Real')
        
        # Add a zero line for reference
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        
        # Adjusting x-axis ticks
        plt.xticks(np.arange(start=450, stop=1701, step=tick_interval))
        plt.legend()
        set_explicit_text_properties(fig_diff)
        plt.tight_layout()
        
        # Filename for difference plot
        diff_filename = f"{RESULT_FOLDER}/{DATE} - {safe_label}_Concentration_{safe_conc}_enhanced_difference.png"
        plt.savefig(diff_filename, dpi=300)
        plt.show()
        plt.close('all')
        plt.clf()
        plt.cla()

# Main processing loop for all virus samples
def process_all_virus_samples(predicted_df, grouped_df_fixed, highest_single_virus_conc_df, reference_series, real_lookup_dict):
    """
    Process all virus samples, creating both standard comparison plots and enhanced plots.
    
    Returns:
        List of MAE records for each processed sample
    """
    global mae_records  # Using the global mae_records list
    mae_records = []
    
    print("\nProcessing all virus samples...")
    # Get all unique virus labels
    all_virus_labels = predicted_df['Label'].unique()
    
    # Categorize labels
    single_virus_labels = [label for label in all_virus_labels if isinstance(label, str) and not "'__'" in label]
    mixed_virus_labels = [label for label in all_virus_labels if isinstance(label, str) and "'__'" in label]
    
    print(f"Found {len(single_virus_labels)} single virus labels and {len(mixed_virus_labels)} mixed virus labels")
    
    # Process single virus samples first
    print("\nProcessing single virus samples...")
    for label in single_virus_labels:
        print(f"Working on {label}")
        # Get concentrations for this label
        conc_values = predicted_df[predicted_df['Label'] == label]['Conc'].unique()
        
        for conc in conc_values:
            conc_str = str(conc)
            print(f"Working on {label}, conc = {conc_str}")
            
            # # Create standard comparison plot
            # create_comparison_plot(label, conc, predicted_df, grouped_df_fixed, real_lookup_dict)
            
            # Create enhanced plot with components
            create_enhanced_plot(label, conc, predicted_df, grouped_df_fixed, 
                                highest_single_virus_conc_df, reference_series, real_lookup_dict)
    
    # Process mixed virus samples
    print("\nProcessing mixed virus samples...")
    for label in mixed_virus_labels:
        print(f"Working on {label}")
        # Get concentrations for this label
        conc_values = predicted_df[predicted_df['Label'] == label]['Conc'].unique()
        
        for conc in conc_values:
            conc_str = str(conc)
            print(f"Working on {label}, conc = {conc_str}")
            
            # Create standard comparison plot
            create_comparison_plot(label, conc, predicted_df, grouped_df_fixed, real_lookup_dict)
            
            # Create enhanced plot with components
            create_enhanced_plot(label, conc, predicted_df, grouped_df_fixed, 
                                highest_single_virus_conc_df, reference_series, real_lookup_dict)
    
    return mae_records

# Function to set explicit text properties for heatmaps
def set_explicit_text_properties_for_heatmap(fig):
    # Set font sizes for all text elements
    for ax in fig.axes:
        ax.title.set_fontsize(24)  # Title font size
        ax.xaxis.label.set_fontsize(22)  # X-axis label font size
        ax.yaxis.label.set_fontsize(22)  # Y-axis label font size
        ax.tick_params(axis='both', which='major', labelsize=20)  # Tick label font size
        
        # Set legend font size if legend exists
        if ax.get_legend() is not None:
            for text in ax.get_legend().get_texts():
                text.set_fontsize(20)

# Function to process difference regions and create analysis visualizations
def analyze_difference_regions(mae_records, predicted_df, grouped_df_fixed):
    """
    Analyze significant difference regions between real and predicted spectra.
    
    Returns:
        List of dictionaries with difference region information
    """
    difference_regions_data = []
    print("\nAnalyzing significant difference regions...")
    
    if not mae_records:
        print("No MAE records to analyze.")
        return difference_regions_data
    
    for record in mae_records:
        label = record['Label']
        conc = record['Concentration']
        
        # Get the data
        real_data = grouped_df_fixed[(grouped_df_fixed['Label'] == label) & (grouped_df_fixed['Conc'] == conc)]
        predicted_data = predicted_df[(predicted_df['Label'] == label) & (predicted_df['Conc'] == conc)]
        
        if real_data.empty or predicted_data.empty:
            continue
        
        # Get wavenumbers and spectra
        wavenumbers = real_data.columns[2:].astype(float)
        real_spectrum = real_data.iloc[0, 2:].values
        predicted_spectrum = predicted_data.iloc[0, 2:].values
        
        # Find regions with significant differences
        regions, differences = find_difference_regions(real_spectrum, predicted_spectrum, wavenumbers)
        
        # Parse virus names from label using ast.literal_eval for reliable parsing
        virus_names = []
        try:
            if isinstance(label, str) and label.startswith('[') and label.endswith(']'):
                try:
                    parsed_label = ast.literal_eval(label)
                    if isinstance(parsed_label, list):
                        virus_names = parsed_label
                    else:
                        virus_names = [str(parsed_label)]
                except (SyntaxError, ValueError):
                    # Fall back to string splitting
                    if "'__'" in label:
                        clean_label = label.strip("[]' ")
                        parts = clean_label.split("'__'")
                        virus_names = [part.strip("'") for part in parts]
                    else:
                        clean_label = label.strip("[]'")
                        virus_names = [clean_label]
            else:
                virus_names = [label]
        except:
            virus_names = [label]  # Default fallback
        
        # Store the difference regions for each virus name
        for virus_name in virus_names:
            for region_start, region_end in regions:
                difference_regions_data.append({
                    'Label': label,
                    'Focus_Virus': virus_name,
                    'Concentration': conc,
                    'Region_Start': region_start,
                    'Region_End': region_end,
                    'MAE_in_Region': np.mean(np.abs(predicted_spectrum[(wavenumbers >= region_start) & 
                                                                    (wavenumbers <= region_end)] - 
                                                real_spectrum[(wavenumbers >= region_start) & 
                                                            (wavenumbers <= region_end)]))
                })
    
    return difference_regions_data

# Function to create difference heatmaps
def create_difference_heatmaps(difference_regions_data, folder):
    """
    Create heatmaps visualizing the differences between real and predicted spectra.
    
    Args:
        difference_regions_data: List of dictionaries with difference region information
        folder: Folder to save the heatmaps
    """
    if not difference_regions_data:
        print("No difference regions data to visualize.")
        return
    
    print("\nCreating difference heatmaps...")
    
    # Group by virus
    virus_groups = {}
    for record in difference_regions_data:
        virus = record['Focus_Virus']
        if virus not in virus_groups:
            virus_groups[virus] = []
        virus_groups[virus].append(record)
    
    # Process each virus group
    for virus, records in virus_groups.items():
        print(f"Creating heatmap for {virus}...")
        
        # Group by label and concentration
        label_conc_groups = {}
        for record in records:
            key = (record['Label'], record['Concentration'])
            if key not in label_conc_groups:
                label_conc_groups[key] = []
            label_conc_groups[key].append(record)
        
        # Sort keys for consistent ordering
        sorted_keys = sorted(label_conc_groups.keys())
        
        # Create a matrix for the heatmap
        # Each row is a label+concentration, each column is a wavenumber
        wavenumbers = np.arange(450, 1701)
        heatmap_matrix = np.zeros((len(sorted_keys), len(wavenumbers)))
        
        # Fill the matrix with MAE values
        for i, key in enumerate(sorted_keys):
            for record in label_conc_groups[key]:
                start_idx = int(record['Region_Start'] - 450)
                end_idx = int(record['Region_End'] - 450)
                if start_idx < 0:
                    start_idx = 0
                if end_idx >= len(wavenumbers):
                    end_idx = len(wavenumbers) - 1
                
                # Set the MAE value for this region
                heatmap_matrix[i, start_idx:end_idx+1] = record['MAE_in_Region']
        
        # Create the heatmap
        fig, ax = plt.subplots(figsize=(12, 8))
        im = ax.imshow(heatmap_matrix, aspect='auto', cmap='viridis', interpolation='nearest')
        
        # Set axis labels
        ax.set_xlabel('Wavenumber')
        ax.set_ylabel('Sample')
        
        # Set y-tick labels
        ax.set_yticks(np.arange(len(sorted_keys)))
        # Create more descriptive y labels
        y_labels = []
        for key in sorted_keys:
            label, conc = key
            # Format label for display
            if isinstance(label, str) and "'__'" in label:
                # For mixed virus, show a simplified format
                clean_label = "Mixed"
            else:
                # For single virus, just show the virus name
                clean_label = "Single"
            y_labels.append(f"{clean_label} {conc}")
        ax.set_yticklabels(y_labels)
        
        # Create custom x-ticks for wavenumbers
        num_ticks = 7  # Number of ticks to show
        tick_indices = np.linspace(0, len(wavenumbers) - 1, num_ticks, dtype=int)
        ax.set_xticks(tick_indices)
        ax.set_xticklabels([str(wavenumbers[i]) for i in tick_indices])
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Mean Absolute Error')
        
        # Set title
        ax.set_title(f'Difference Analysis Heatmap for {virus}')
        
        # Apply text properties
        set_explicit_text_properties_for_heatmap(fig)
        
        # Save the figure
        plt.tight_layout()
        safe_virus = str(virus).replace('/', '_').replace('\\', '_').replace(':', '_')
        plt.savefig(os.path.join(folder, f"{safe_virus}_difference_heatmap.png"), dpi=300)
        plt.show()
        plt.close('all')
        plt.clf()
        plt.cla()

# Function to create MAE summary visualizations
def create_mae_summary_plots(mae_records, folder):
    """
    Create summary plots for MAE values.
    
    Args:
        mae_records: List of dictionaries with MAE information
        folder: Folder to save the plots
    """
    if not mae_records:
        print("No MAE records to visualize.")
        return
    
    print("\nCreating summary plots for MAE values...")
    
    # Convert to DataFrame
    mae_df = pd.DataFrame(mae_records)
    
    # Group by label
    label_groups = mae_df.groupby('Label')
    
    # Create a summary figure for single vs mixed virus MAE
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Categorize labels as single or mixed
    mae_df['Type'] = mae_df['Label'].apply(lambda x: 'Mixed' if "'__'" in str(x) else 'Single')
    
    # Calculate mean MAE for each type
    type_mae = mae_df.groupby('Type')['MAE'].mean()
    
    # Create bar chart
    type_mae.plot(kind='bar', ax=ax, color=['blue', 'orange'])
    ax.set_ylabel('Mean Absolute Error')
    ax.set_title('Average MAE by Virus Type')
    
    # Apply text properties
    set_explicit_text_properties(fig)
    
    # Save the figure
    plt.tight_layout()
    plt.savefig(os.path.join(folder, f"{DATE} - virus_type_mae_summary.png"), dpi=300)
    plt.show()
    plt.close('all')
    plt.clf()
    plt.cla()
    
    # Create individual plots for each virus
    for label, group in label_groups:
        if len(group) > 1:  # Only create plot if multiple concentrations
            print(f"Creating MAE summary plot for {label}...")
            
            # Sort by concentration if possible
            try:
                # Convert concentration to numeric if possible
                group = group.copy()
                group['NumericConc'] = group['Concentration'].apply(
                    lambda x: float(x.strip('[]')) if isinstance(x, str) else float(x))
                group = group.sort_values('NumericConc')
            except:
                # If conversion fails, keep original order
                pass
            
            # Create the plot
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.plot(range(len(group)), group['MAE'], 'o-', linewidth=2, markersize=8)
            
            # Set tick labels
            ax.set_xticks(range(len(group)))
            ax.set_xticklabels(group['Concentration'], rotation=45)
            
            # Set labels and title
            ax.set_xlabel('Concentration')
            ax.set_ylabel('Mean Absolute Error')
            
            # Format label for title
            display_label = label
            if isinstance(label, str) and label.startswith('[') and label.endswith(']'):
                # Clean up the label for display
                if "'__'" in label:
                    # Mixed virus case, format with +
                    clean_label = label.strip("[]' ")
                    parts = clean_label.split("'__'")
                    display_label = " + ".join([part.strip("'") for part in parts])
                else:
                    # Single virus case
                    display_label = label.strip("[]'")
            
            ax.set_title(f'MAE vs. Concentration for {display_label}')
            
            # Apply text properties
            set_explicit_text_properties(fig)
            
            # Save the figure
            plt.tight_layout()
            safe_label = str(label).replace('/', '_').replace('\\', '_').replace(':', '_')
            plt.savefig(os.path.join(folder, f"{DATE} - {safe_label}_mae_summary.png"), dpi=300)
            plt.show()
            plt.close('all')
            plt.clf()
            plt.cla()


In [55]:
# Fix data anomalies and prepare lookup dictionaries
grouped_df_fixed, real_lookup_dict = fix_and_prepare_data()
print("Fixed DataFrame shape:", grouped_df_fixed.shape)
grouped_df_fixed



Fixing data anomalies and preparing lookup dictionaries...
No data anomalies found, using existing data.
Fixed DataFrame shape: (2580, 1253)


,Label,Conc,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,...,1691.0,1692.0,1693.0,1694.0,1695.0,1696.0,1697.0,1698.0,1699.0,1700.0
0,['Ad5'],[100.0],0.188456,0.193579,0.199298,0.205842,0.214035,0.222088,0.230509,0.238614,...,0.724246,0.706123,0.688772,0.672316,0.656649,0.642035,0.628474,0.616544,0.603561,0.589719
1,['Ad5'],[100000.0],0.108017,0.115627,0.123576,0.132051,0.140915,0.149695,0.160068,0.166644,...,1.058237,1.045763,1.033407,1.020407,1.007390,0.992627,0.976119,0.957441,0.937000,0.914610
2,['Ad5'],[12500.0],0.125390,0.136017,0.145695,0.155153,0.164305,0.172593,0.181102,0.186508,...,1.031186,1.019051,1.007000,0.994695,0.982814,0.969153,0.953915,0.936441,0.918119,0.898695
3,['Ad5'],[1562.0],0.144458,0.155678,0.165746,0.175237,0.184034,0.191864,0.199475,0.204407,...,1.011746,1.001780,0.990966,0.979593,0.967610,0.954356,0.939966,0.923898,0.907797,0.891576
4,['Ad5'],[195.0],0.142603,0.152603,0.161483,0.169983,0.177655,0.184931,0.192069,0.197793,...,0.880224,0.866776,0.852776,0.838741,0.824276,0.809707,0.795000,0.779914,0.764931,0.750431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2575,['RSVB1'],[50.0],0.136810,0.138224,0.141741,0.146172,0.151672,0.158603,0.166310,0.174000,...,0.744603,0.726397,0.707000,0.688672,0.671207,0.653741,0.637914,0.623000,0.611241,0.598483
2576,['RSVB1'],[50000.0],0.154390,0.162458,0.167458,0.171000,0.172881,0.173220,0.170322,0.170288,...,0.860475,0.851661,0.843661,0.836492,0.831085,0.823475,0.813983,0.801017,0.789203,0.778390
2577,['RSVB1'],[6250.0],0.177224,0.182293,0.185483,0.187879,0.189190,0.189483,0.187948,0.187966,...,0.815948,0.802086,0.791810,0.783845,0.781293,0.775534,0.767638,0.754776,0.744207,0.735483
2578,['RSVB1'],[781.0],0.126759,0.135397,0.141224,0.145879,0.149379,0.151500,0.151138,0.153155,...,0.735293,0.720603,0.709017,0.699397,0.694690,0.686190,0.675052,0.658069,0.643534,0.630948


In [ ]:
# Process all virus samples (creates both standard comparison and enhanced plots)
mae_records = process_all_virus_samples(predicted_df, grouped_df_fixed, highest_single_virus_conc_df, 
                                      reference_series, real_lookup_dict)
mae_records

In [57]:
# Save MAE records to CSV
if mae_records:
    mae_df = pd.DataFrame(mae_records)
    mae_csv_path = os.path.join(R2_MAE_FOLDER, f"{DATE} - reconstruction_mae_summary.csv")
    mae_df.to_csv(mae_csv_path, index=False)
    print(f"MAE summary saved to {mae_csv_path}")
else:
    print("No MAE records collected, skipping MAE summary CSV.")

MAE summary saved to /home/zhao/Jiaheng Cui/Coefficient_fitting/04042025 - r2_mae_analysis/04042025 - reconstruction_mae_summary.csv
